# Urdu-Seg: CRF baseline

Trains the character-level CRF of Section 5.1 with five-fold cross-validation (clean and noisy text, 10 models)
and writes `results/crf/pred_crf_clean.csv` and `results/crf/pred_crf_noisy.csv`. Takes roughly 10–30 minutes.
Run it from anywhere inside the cloned repository; afterwards `python reproduce.py` recomputes Table 11.

In [ ]:
# 1) Find the repository folder (works wherever the notebook is opened)
import os, sys, glob
def _find_root():
    here = os.getcwd()
    for s in [here, os.path.dirname(here), os.path.dirname(os.path.dirname(here)), os.path.expanduser("~")]:
        for c in [s] + glob.glob(os.path.join(s, "*")) + glob.glob(os.path.join(s, "*", "*")) + glob.glob(os.path.join(s, "*", "*", "*")):
            if os.path.isfile(os.path.join(c, "src", "crf_baseline.py")):
                return os.path.abspath(c)
    raise FileNotFoundError("Could not find the repository folder (the one that contains src/crf_baseline.py)")
ROOT = _find_root(); os.chdir(ROOT); sys.path.insert(0, os.path.join(ROOT, "src"))
print("working in", ROOT)

In [ ]:
# 2) Install the CRF library
%pip install python-crfsuite

In [ ]:
# 3) Train and predict (prints one line per fold)
import subprocess
env = dict(os.environ, PYTHONIOENCODING="utf-8", PYTHONUTF8="1")
p = subprocess.Popen([sys.executable, "src/crf_baseline.py", "data", "results/crf"], stdout=subprocess.PIPE,
                     stderr=subprocess.STDOUT, text=True, encoding="utf-8", env=env)
for line in p.stdout:
    print(line, end="")
p.wait(); print("exit code:", p.returncode)

In [ ]:
# 4) Quick look at the scores
for name, gold, pred in (("independent, clean", "data/v3/v2dedup_revised.csv", "results/crf/pred_crf_clean.csv"),
                         ("full, clean", "data/v3/gold_v3.csv", "results/crf/pred_crf_clean.csv"),
                         ("full, noisy", "data/v3/gold_v3_noisy.csv", "results/crf/pred_crf_noisy.csv")):
    r = subprocess.run([sys.executable, "src/evaluate.py", "--gold", gold, "--split", "test", "--bootstrap", "0",
                        "--pred", "CRF=" + pred], capture_output=True, text=True, encoding="utf-8", env=env)
    print("==", name); print(r.stdout or r.stderr)